In [8]:
import pandas as pd
from Bio import SeqIO
from modules.sequences import split_train_val_test

from modules.model import (
    init_transition_mat,
    remove_silent_states,
    init_emission_mat
)

from modules.train import train_phmm
from modules.evaluation import description_accuracy
from modules.model import build_hmmlearn_model

from pathlib import Path
import logging
import joblib

import numpy as np

from modules.evaluation import (
    plot_da_learning_curve,
    plot_da_bars,
)
import matplotlib.pyplot as plt

In [9]:
# Read protein families
family_df = pd.read_csv("protein_families.csv")

i = 0
name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]
match_len = int(match_len)

###### SETUP
settings = {
    "n=n": match_len,
    "n=n-5": match_len - 5,
    "n=n-10": match_len - 10,
}

trans_strength = 1.0
emit_strength = 5.0

max_iter = 15
tol = 1e-3

# Directories for saved models and results
model_dir = Path("saved_models/phmm/n")
results_dir = Path("results/phmm/n")

model_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
######

logging.getLogger("hmmlearn").setLevel(logging.ERROR)

# Read FASTA files
recs = [
    (r.id, str(r.seq))
    for r in SeqIO.parse(
        f"sequences/{accession}.fasta", "fasta"
    )
]

# Same train/validation/test split for all configurations
train_seqs, val_seqs, test_seqs = split_train_val_test(recs)

results = {}
histories = {}
trained_models = {}

for label, model_match_len in settings.items():
    model_match_len = int(model_match_len)

    if model_match_len < 2:
        raise ValueError(
            f"Invalid match length: {model_match_len}"
        )

    print(
        f"\n--- {label}: "
        f"match_len={model_match_len}, "
        f"trans_strength={trans_strength}, "
        f"emit_strength={emit_strength}",
        flush=True
    )

    # 1. Initialize matrices for this match length
    trans_matdf = init_transition_mat(model_match_len)
    emit_matdf = init_emission_mat(
        model_match_len, seed=2
    )

    # 2. Remove silent D states
    cols = list(trans_matdf.columns)
    d_cols = [c for c in cols if c.startswith("D")]

    trans_mat, red_cols = remove_silent_states(
        trans_matdf.to_numpy(),
        cols,
        d_cols
    )

    trans_matdf = pd.DataFrame(
        trans_mat,
        columns=red_cols,
        index=red_cols
    )

    # 3. Convert matrices to hmmlearn convention
    A = trans_mat.T
    B = emit_matdf.to_numpy().T

    startprob = np.zeros(A.shape[0])
    startprob[0] = 1.0

    # 4. Encode symbols
    symbols = list(emit_matdf.index)
    sym_idx = {
        s: j for j, s in enumerate(symbols)
    }

    BEGIN = sym_idx["^"]
    END = sym_idx["$"]

    # Number of hidden states
    n_states = A.shape[0]

    # 5. Build untrained PHMM
    model_initial = build_hmmlearn_model(
        A, B, startprob,
        trans_strength=trans_strength,
        emit_strength=emit_strength
    )

    # 6. Evaluate untrained PHMM
    da0 = description_accuracy(
        model_initial, sym_idx, test_seqs
    )

    untrained_mean = float(da0.mean())
    untrained_std = float(da0.std())

    # 7. Train PHMM
    model, hist = train_phmm(
        model_initial,
        sym_idx,
        train_seqs,
        val_seqs,
        None,
        max_iter,
        tol,
    )

    histories[label] = hist
    trained_models[label] = model

    # 8. Save model immediately after training
    prefix = (
        f"{accession}_m{model_match_len}"
        f"_ts{trans_strength:g}_es{emit_strength:g}"
    )

    model_path = model_dir / f"{prefix}.joblib"
    history_path = results_dir / f"{prefix}_history.csv"

    joblib.dump(model, model_path)
    hist.to_csv(history_path, index=False)

    print(f"Saved model: {model_path}", flush=True)

    # 9. Evaluate trained PHMM on test set
    da = description_accuracy(
        model, sym_idx, test_seqs
    )

    trained_mean = float(da.mean())
    trained_std = float(da.std())

    # 10. Save results
    results[label] = {
        "match_len": model_match_len,
        "hidden_states": n_states,
        "trans_strength": trans_strength,
        "emit_strength": emit_strength,
        "untrained_da_mean": untrained_mean,
        "untrained_da_std": untrained_std,
        "trained_da_mean": trained_mean,
        "trained_da_std": trained_std,
        "epochs_completed": len(hist) - 1
    }

    print(
        f"Untrained DA: {untrained_mean:.4f} "
        f"± {untrained_std:.4f}\n"
        f"Trained DA:   {trained_mean:.4f} "
        f"± {trained_std:.4f}",
        flush=True
    )

    # Save combined results after each configuration
    pd.DataFrame.from_dict(
        results, orient="index"
    ).rename_axis("setting").to_csv(
        results_dir / f"{accession}_match_len_comparison.csv"
    )

print("\nAll PHMM configurations completed.")


--- n=n: match_len=39, trans_strength=1.0, emit_strength=5.0
iter   0  cpu     0.0s  log-lik -14656.90  train_da_mean 0.0023  val_da_mean 0.0033
iter   1  cpu     0.1s  log-lik -13179.05  train_da_mean 0.1080  val_da_mean 0.1024
iter   2  cpu     0.2s  log-lik -11735.58  train_da_mean 0.2165  val_da_mean 0.1951
iter   3  cpu     0.3s  log-lik -10783.38  train_da_mean 0.2835  val_da_mean 0.2497
iter   4  cpu     0.5s  log-lik -10424.78  train_da_mean 0.3093  val_da_mean 0.2689
iter   5  cpu     0.6s  log-lik -10160.26  train_da_mean 0.3271  val_da_mean 0.2830
iter   6  cpu     0.7s  log-lik -10000.83  train_da_mean 0.3373  val_da_mean 0.2911
iter   7  cpu     0.8s  log-lik -9898.05  train_da_mean 0.3438  val_da_mean 0.2962
iter   8  cpu     0.9s  log-lik -9831.22  train_da_mean 0.3483  val_da_mean 0.3003
iter   9  cpu     1.0s  log-lik -9797.55  train_da_mean 0.3505  val_da_mean 0.3018
iter  10  cpu     1.2s  log-lik -9779.83  train_da_mean 0.3516  val_da_mean 0.3027
iter  11  cpu     